# Pertemuan 3: Data Cleaning : Missing Values, Outlier & Ekstrasi Data

| | |
|---|---|
| **Nama Lengkap** | Nisa Agustina Maesaroh |
| **NIM** | 240401070509 |
| **Kelas** | IF403 |


In [310]:
# ================================================
# PIPELINE DATA CLEANING - HOUSING DATASET
# ================================================

import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import requests
from pandas import json_normalize

print('Library berhasil diimport!')

Library berhasil diimport!


In [311]:
# STEP 0 - Load & eksplorasi awal
df = pd.read_csv('dataset/housing_dirty.csv')
print('--- HEAD (5 Baris Pertama) ---')
df.head()

--- HEAD (5 Baris Pertama) ---


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


In [312]:
#info dataset
print('--- INFO DATASET ---')
print(f'Shape awal:', df.shape)
print()
df.info()

#statistik deskriptif
print('\n--- STATISTIK DESKRIPTIF ---')
df.describe()

--- INFO DATASET ---
Shape awal: (130, 7)

<class 'pandas.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    str    
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    str    
dtypes: float64(3), int64(2), str(2)
memory usage: 7.2 KB

--- STATISTIK DESKRIPTIF ---


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [313]:
#cek missing values
print('--- MISSING VALUES PER KOLOM ---')
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Jumlah Missing' : missing_count,
    'Persentase (%)' : missing_pct
})
print(missing_summary)
print(f'\nTotal nilai missing: {df.isnull().sum().sum()}')

--- MISSING VALUES PER KOLOM ---
              Jumlah Missing  Persentase (%)
id                         0            0.00
luas_m2                   18           13.85
harga_juta                17           13.08
kota                       0            0.00
kamar                     10            7.69
tahun_bangun               0            0.00
kondisi                    0            0.00

Total nilai missing: 45


In [314]:
#STEP 1 - Hapus duplikat
print('--- Cek Duplikat ---')
n_dup = df.duplicated().sum()
print(f'Jumlah baris duplikat: {n_dup} dari {len(df)} total baris')

--- Cek Duplikat ---
Jumlah baris duplikat: 0 dari 130 total baris


In [315]:
#tampilkan baris sebelum dihapus
df_dup = df[df.duplicated(keep=False)]
print(f'Baris duplikat yang ditemukan:')
print(df_dup)

df.drop_duplicates(inplace=True)
print(f'Shape setelah hapus duplikat:', df.shape)
print(f'Sisa duplikat:', df.duplicated().sum())

Baris duplikat yang ditemukan:
Empty DataFrame
Columns: [id, luas_m2, harga_juta, kota, kamar, tahun_bangun, kondisi]
Index: []
Shape setelah hapus duplikat: (130, 7)
Sisa duplikat: 0


In [316]:
#STEP 2 - Normalisasi string
#cek nilai sebelum normalisasi
print('Nilai unik kolom [kota] SEBELUM normalisasi:')
print(df['kota'].unique())

print('\nNilai unik kolom [kondisi] SEBELUM normalisasi:')
print(df['kondisi'].unique())

Nilai unik kolom [kota] SEBELUM normalisasi:
<StringArray>
[     'jogja',      'Medan',      'Depok',        'YGY',    'Jakarta',
    'jakarta', 'Yogyakarta',    'Bandung',   'Surabaya',        'dpk',
        'sby',   'Makassar',        'mdn',      'medan',   'Semarang',
   'semarang', 'yogyakarta',      'Jogja',    'JAKARTA',        'Smg',
      'DEPOK',        'Bdg',   'makassar',   'surabaya',   'MAKASSAR',
      'depok',    'bandung',   'Bandung ',   'SURABAYA',       'Mksr',
   ' Jakarta']
Length: 31, dtype: str

Nilai unik kolom [kondisi] SEBELUM normalisasi:
<StringArray>
[          'baik',          'Bagus',         'Sedang',    'baik sekali',
         'SEDANG',         'sedang',           'BAIK',          'rusak',
          'cukup',           'Baik',          'Cukup', 'perlu renovasi',
          'bagus',          'jelek',          'RUSAK']
Length: 15, dtype: str


In [317]:
#normalisasi
df['kota'] = df['kota'].str.strip().str.title() #strip whitespace + title case
df['kondisi'] = df['kondisi'].str.strip().str.lower() #strip whitespace + lowercase
print('nilai unik kolom [kota] SETELAH normalisasi')
print(df['kota'].unique())

print('\nnilai unik kolom [kondisi] SETELAH normalisasi')
print(df['kondisi'].unique())

nilai unik kolom [kota] SETELAH normalisasi
<StringArray>
[     'Jogja',      'Medan',      'Depok',        'Ygy',    'Jakarta',
 'Yogyakarta',    'Bandung',   'Surabaya',        'Dpk',        'Sby',
   'Makassar',        'Mdn',   'Semarang',        'Smg',        'Bdg',
       'Mksr']
Length: 16, dtype: str

nilai unik kolom [kondisi] SETELAH normalisasi
<StringArray>
[          'baik',          'bagus',         'sedang',    'baik sekali',
          'rusak',          'cukup', 'perlu renovasi',          'jelek']
Length: 8, dtype: str


In [318]:
#STEP 3 - Imputasi missing values
median_luas = df['luas_m2'].median()
median_harga = df['harga_juta'].median()
modus_kamar = df['kamar'].mode()[0]

print(f'Nilai imputasi luas_m2: {median_luas} (median)')
print(f'Nilai imputasi harga_juta: {median_harga} (median)')
print(f'Nilai imputasi kamar: {modus_kamar} (modus)')

#imputasi
df['luas_m2'] = df['luas_m2'].fillna(median_luas)
df['harga_juta'] = df['harga_juta'].fillna(median_harga)
df['kamar'] = df['kamar'].fillna(modus_kamar)

print(f'\nMissing values setelah imputasi: {df.isnull().sum().sum()}')

Nilai imputasi luas_m2: 193.8 (median)
Nilai imputasi harga_juta: 655.0 (median)
Nilai imputasi kamar: 1.0 (modus)

Missing values setelah imputasi: 0


In [319]:
#STEP 4 - Tangani Outlier (IQR Fence)
#fungsi deteksi outlier
def deteksi_outlier_iqr(df, kolom):
    Q1 = df[kolom].quantile(0.25)
    Q3 = df[kolom].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[kolom] < lower) | (df[kolom] > upper)]
    return lower, upper, outliers

# deteksi & tangani outlier untuk kolom target
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    lower, upper, outliers = deteksi_outlier_iqr(df, col)
    print(f'[{col}] — Batas IQR: [{lower:.2f}, {upper:.2f}] | Outlier: {len(outliers)} baris')

    # clip (capping) nilai outlier ke batas IQR
    df[col] = df[col].clip(lower=lower, upper=upper)

print('\nOutlier berhasil ditangani dengan IQR Fence (capping).')

[harga_juta] — Batas IQR: [-422.75, 1719.25] | Outlier: 3 baris
[luas_m2] — Batas IQR: [-145.22, 512.97] | Outlier: 1 baris
[tahun_bangun] — Batas IQR: [1960.50, 2042.50] | Outlier: 3 baris

Outlier berhasil ditangani dengan IQR Fence (capping).


In [320]:
#verifikasi statistik setelah capping
print('--- STATISTIK SETELAH PENANGANAN OUTLIER ---')
df[['harga_juta', 'luas_m2', 'tahun_bangun']].describe()

--- STATISTIK SETELAH PENANGANAN OUTLIER ---


,harga_juta,luas_m2,tahun_bangun
count,130.000000,130.000000,130.000000
mean,686.490385,188.274423,2001.542308
std,404.633957,95.297150,13.505818
min,-422.750000,-50.000000,1960.500000
25%,380.500000,101.600000,1991.250000
50%,655.000000,193.800000,2002.000000
75%,916.000000,266.150000,2011.750000
max,1719.250000,512.975000,2042.500000


In [321]:
#STEP 5 - Validasi & ekspor
#validasi
print('--- VALIDASI AKHIR ---')

total_missing = df.isnull().sum().sum()
total_dup     = df.duplicated().sum()

print(f'Total missing values : {total_missing}' if total_missing == 0 else f'Total missing values : {total_missing}  ✗ GAGAL!')
print(f'Total duplikat       : {total_dup}' if total_dup == 0 else f'Total duplikat       : {total_dup}  ✗ GAGAL!')
print(f'Shape akhir          : {df.shape}')

# Assert untuk memastikan kedua kondisi terpenuhi
assert total_missing == 0, 'Masih ada missing values!'
assert total_dup == 0,     'Masih ada duplikat!'

print('\nSemua validasi LULUS!')
# assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
# assert df.duplicated().sum() == 0, 'Masih ada duplikat!'
# print('Shape akhir:', df.shape)
# df.to_csv('housing_clean.csv', index=False)
# print('Dataset bersih tersimpan!')

--- VALIDASI AKHIR ---
Total missing values : 0
Total duplikat       : 0
Shape akhir          : (130, 7)

Semua validasi LULUS!


In [322]:
# Ekspor dataset bersih
df.to_csv('dataset/housing_clean.csv', index=False)
print('Dataset bersih tersimpan sebagai housing_clean.csv')

# Preview hasil akhir
print('\n--- PREVIEW DATASET BERSIH (5 baris pertama) ---')
df.head()

Dataset bersih tersimpan sebagai housing_clean.csv

--- PREVIEW DATASET BERSIH (5 baris pertama) ---


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,Jogja,2.0,2000.0,baik
1,2,254.0,761.0,Medan,1.0,1995.0,bagus
2,3,249.7,895.0,Depok,1.0,1983.0,baik
3,4,49.7,178.0,Ygy,5.0,2013.0,baik
4,5,133.4,424.0,Medan,5.0,2004.0,sedang


In [323]:
# Akses API JSONPlaceholder — endpoint /posts
postUrl = 'https://jsonplaceholder.typicode.com/posts'

try:
    response = requests.get(postUrl, timeout=10)

    if response.status_code == 200:
        data_posts = response.json()
        df_posts   = json_normalize(data_posts, sep='_')
        print(f'Berhasil mengakses API! Status: {response.status_code}')
        print(f'Jumlah data: {len(df_posts)} baris, {len(df_posts.columns)} kolom')
    else:
        print(f'Gagal akses API. Status code: {response.status_code}')

except requests.exceptions.ConnectionError as e:
    print(f'Error koneksi: {e}')
except requests.exceptions.Timeout:
    print('Request timeout — server tidak merespons dalam 10 detik')

Berhasil mengakses API! Status: 200
Jumlah data: 100 baris, 4 kolom


In [324]:
# Tampilkan kolom yang tersedia
print('Kolom yang tersedia:')
print(df_posts.columns.tolist())

# Tampilkan kolom-kolom yang relevan
kolom_pilihan = ['userId', 'id', 'title', 'body']
print('\n--- DATA POSTS DARI JSONPLACEHOLDER API ---')
df_posts[kolom_pilihan]

Kolom yang tersedia:
['userId', 'id', 'title', 'body']

--- DATA POSTS DARI JSONPLACEHOLDER API ---


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...
...,...,...,...,...
95,10,96,quaerat velit veniam amet cupiditate aut numqu...,in non odio excepturi sint eum\nlabore volupta...
96,10,97,quas fugiat ut perspiciatis vero provident,eum non blanditiis soluta porro quibusdam volu...
97,10,98,laboriosam dolor voluptates,doloremque ex facilis sit sint culpa\nsoluta a...
98,10,99,temporibus sit alias delectus eligendi possimu...,quo deleniti praesentium dicta non quod\naut e...


# Kesimpulan

## Apa yang Dipelajari
Mempelajari teknik data cleaning pada dataset kotor (housing_dirty.csv): menangani missing values, outlier dengan winsorize, dan mengambil data dari API menggunakan requests + json_normalize.

## Temuan Utama
Data nyata hampir selalu kotor — duplikat, tipe data salah, dan outlier ekstrem adalah masalah umum. Winsorize lebih aman dari drop untuk outlier karena tidak membuang baris.

## Keterbatasan & Pertanyaan
API yang digunakan (JSONPlaceholder) bersifat dummy, bukan data real. Pertanyaan: bagaimana menangani API yang memerlukan autentikasi (OAuth, API key)?
